# fp16 merge + eval (CLEAN process, NO unsloth -> avoids the apply_qkv patch clash)
Loads Qwen3-1.7B fp16 + the trained adapter, merges, evals on FULL 200 gold,
gates for corruption, pushes merged. Baseline: nested 58%, flat 87%.

In [ ]:
%pip install -q "transformers>=4.55,<5" peft accelerate "jsonschema[format]" datasets
!pip uninstall -y -q torchao  # Kaggle ships 0.10; new peft needs >=0.16 or absent

In [ ]:
import os, json, re, torch
os.environ["HF_TOKEN"]=os.environ.get("HF_TOKEN", "")
HF_BASE="Qwen/Qwen3-1.7B"; ADAPTER="ujwal00/qwen3-1.7b-json-lora"; MERGED="ujwal00/qwen3-1.7b-json"
from jsonschema import Draft202012Validator
FMT=Draft202012Validator.FORMAT_CHECKER
SYSTEM=("You are a precise JSON generator. Given a JSON Schema, output ONE JSON value "
  "that strictly validates against it. Output ONLY the JSON value - no prose, no "
  "markdown fences, no comments. Include every required property, use only the "
  "properties and types the schema allows, and respect enums, formats, and nesting.")
USER_T="Generate a JSON value that validates against this JSON Schema:\n\n{schema}\n\nReturn only the JSON value."
def render(s):
    if isinstance(s,str): s=json.loads(s)
    return [{"role":"system","content":SYSTEM},{"role":"user","content":USER_T.format(schema=json.dumps(s,sort_keys=True,ensure_ascii=False))}]
_F=re.compile(r"^```(?:json)?\s*|\s*```$",re.M)
def extract_json(text):
    t=_F.sub("",text).strip()
    try: return json.loads(t),True
    except Exception: pass
    st=None
    for i,c in enumerate(t):
        if c in "{[": st=i;break
    if st is None: return None,False
    op=t[st];cl="}" if op=="{" else "]";d=0;ins=False;esc=False
    for j in range(st,len(t)):
        c=t[j]
        if ins:
            if esc:esc=False
            elif c=='\\':esc=True
            elif c=='"':ins=False
        else:
            if c=='"':ins=True
            elif c==op:d+=1
            elif c==cl:
                d-=1
                if d==0:
                    try:return json.loads(t[st:j+1]),True
                    except Exception:return None,False
    return None,False
def valid(inst,s):
    if isinstance(s,str): s=json.loads(s)
    try: return len(list(Draft202012Validator(s,format_checker=FMT).iter_errors(inst)))==0
    except Exception: return False

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
tok=AutoTokenizer.from_pretrained(ADAPTER,token=os.environ["HF_TOKEN"])
base=AutoModelForCausalLM.from_pretrained(HF_BASE,dtype=torch.float16,device_map="auto",token=os.environ["HF_TOKEN"])
base.resize_token_embeddings(len(tok))
m=PeftModel.from_pretrained(base,ADAPTER,token=os.environ["HF_TOKEN"]); m=m.merge_and_unload(); m.eval()
if tok.pad_token is None: tok.pad_token=tok.eos_token
tok.padding_side="left"
print("merged fp16 OK")

In [ ]:
from datasets import load_dataset
gold=list(load_dataset("ujwal00/jsonschema-gold",split="train",token=os.environ["HF_TOKEN"]))
@torch.no_grad()
def ev(rows):
    prompts=[tok.apply_chat_template(render(g["schema"]),tokenize=False,add_generation_prompt=True,enable_thinking=False) for g in rows]
    outs=[]
    for i in range(0,len(prompts),16):
        enc=tok(prompts[i:i+16],return_tensors="pt",padding=True,truncation=True,max_length=2048).to(m.device)
        g=m.generate(**enc,max_new_tokens=512,do_sample=False,pad_token_id=tok.eos_token_id)
        outs+=tok.batch_decode(g[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
    fo=fn=no=nn=pf=0; corrupt=0
    for r,t in zip(rows,outs):
        nested=r["complexity_class"]=="nested"
        if nested:nn+=1
        else:fn+=1
        if re.search(r'\}\}\}|<tool_call>',t): corrupt+=1
        obj,ok=extract_json(t)
        if not ok: pf+=1; continue
        if valid(obj,r["schema"]):
            if nested:no+=1
            else:fo+=1
    return {"flat":round(fo/max(fn,1),4),"nested":round(no/max(nn,1),4),"all":round((fo+no)/len(rows),4),"parse_fail":round(pf/len(rows),4),"corrupt":corrupt}
res=ev(gold)
print("MERGED full-gold(200):",json.dumps(res))
print("BASELINE was: nested 0.58 flat 0.87")

In [ ]:
# GATE: no corruption + beats baseline nested -> push merged
gate = res["parse_fail"]<=0.10 and res["nested"]>=0.58  # validator is truth; corrupt is just informational
json.dump(res,open("/kaggle/working/merged_gold.json","w"),indent=2)
print("GATE:", "PASS" if gate else "FAIL", res)
assert gate, "gate failed - not pushing"
m.push_to_hub(MERGED,token=os.environ["HF_TOKEN"],private=True,max_shard_size="4GB")
tok.push_to_hub(MERGED,token=os.environ["HF_TOKEN"],private=True)
print("pushed merged:",MERGED)